# Develop

## Import

In [1]:
import numpy as np
import pandas as pd
import json

import pyarrow as pa
import pyarrow.parquet as pq

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, row_number, lit

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score

from typing import List

## Model learning

In [2]:
def train_in_chunks(files_list: List[str] = None,
                    cols_json_file: str = None,
                    batchsize: int = 20000,
                    model: CatBoostClassifier = None,
                    auto_nan_filler: bool = True):
    """
    Функция для последовательного обучения модели по чанкам. Принимает следующие аргументы:
    files_list - список с файлами для обучения модели (полные пути до файлов)
    cols_json_file - путь до json файла с инфой по колонкам
    batchsize - размер чанка (кол-во строк)
    model - модель
    """
    if files_list is None:
        raise AttributeError("Надо указать files_list - файл(ы) для обучения. Обязательно в списке")
    if cols_json_file is None:
        raise AttributeError("Надо указать cols_json_file - json файл с инфой по колонкам")
    if model is None:
        raise AttributeError("Надо указать model - действующую модель")
    
    with open(cols_json_file, 'r') as file:
        cols = json.load(file)
    
    feature_cols = cols['all']
    cat_cols = cols['cat']
    target_col = cols['target']
    
    for file in files_list:
        current_file = pq.ParquetFile(file)
        total_rows = current_file.metadata.num_rows
        total_batches = (total_rows + batchsize - 1) // batchsize
        print(f'В файле {file} {total_rows} строк. Это {total_batches} батчей')
        
        for batch_num, batch in enumerate(current_file.iter_batches(batch_size=batchsize)):
            print(f'Старт обработки {batch_num + 1}/{total_batches} батча')
            chunk = batch.to_pandas()
            
            print('В target батча содержится:')
            for value in chunk[target_col].unique():
                print(f'{value} - {len(chunk[chunk[target_col] == value])} штук')
            
            if auto_nan_filler:
                for col in cat_cols:
                    if col in chunk.columns:
                        chunk[col] = chunk[col].fillna('NaN').astype(str)
            
            X_batch = chunk[feature_cols]
            y_batch = chunk[target_col]
            train_pool = Pool(X_batch, y_batch, cat_features=cat_cols)
            
            print(f'Старт обучения {batch_num + 1}/{total_batches} батча')
            if batch_num == 0:
                model.fit(train_pool)
            else:
                model.fit(train_pool, init_model=model)
            print(f'Обучение {batch_num + 1}/{total_batches} батча завершено')
            print('==========================================================')

    model.save_model('../models/catboost_model.cbm')
    print(f'Обучение полностью завершено. Модель сохранена')

In [3]:
valid_train_data_path = '../datasets/joined/train_data.parquet'

catboost_model = CatBoostClassifier(verbose=0)

train_in_chunks(files_list=[valid_train_data_path],
                cols_json_file='../datasets/joined/columns_list.json',
                model=catboost_model
                )

В файле ../datasets/joined/train_data.parquet 91891 строк. Это 5 батчей
Старт обработки 1/5 батча
В target батча содержится:
0 - 19959 штук
1 - 41 штук
Старт обучения 1/5 батча
Обучение 1/5 батча завершено
Старт обработки 2/5 батча
В target батча содержится:
0 - 19937 штук
1 - 63 штук
Старт обучения 2/5 батча
Обучение 2/5 батча завершено
Старт обработки 3/5 батча
В target батча содержится:
0 - 19949 штук
1 - 51 штук
Старт обучения 3/5 батча
Обучение 3/5 батча завершено
Старт обработки 4/5 батча
В target батча содержится:
0 - 19944 штук
1 - 56 штук
Старт обучения 4/5 батча
Обучение 4/5 батча завершено
Старт обработки 5/5 батча
В target батча содержится:
0 - 11870 штук
1 - 21 штук
Старт обучения 5/5 батча
Обучение 5/5 батча завершено
Обучение полностью завершено. Модель сохранена


## Prediction

In [4]:
def save_file_results(id_events: pd.Series,
                      predictions: np.ndarray,
                      save_path: str,
                      chunk_num: int):
    """
    Сохранение предсказаний вместе с идентификаторами в Parquet файл
    Принимает:
    id_events - объект pd.Series, который формируется в функции predict_in_chunks.
        Содержит данные об id операций. Нужен для коммита
    predictions - массив нампая с предсказаниями
    save_path - полный путь сохранения parquet файла
    chunk_num - флаговая переменная для указания номера чанка.
        Если чанк под номером 0 (первый), то надо создать новую таблицу. Иначе выполнить соединение с существующей
    """
    result_df = pd.DataFrame({
        'event_id': id_events,
        'predict': predictions
    })
    
    # Конвертируем в Arrow Table
    table = pa.Table.from_pandas(result_df)
    with open(save_path, 'w') as f:
        if chunk_num == 0:
            # Первый чанк или файл не существует - создаем новый
            pq.write_table(
                table,
                save_path,
                compression='snappy',
                flavor='spark'  # для совместимости со Spark если нужно
            )
            print(f"Создан файл: {save_path} ({len(result_df)} записей)")
        else:
            # Добавляем данные в существующий файл
            with pq.ParquetWriter(
                save_path,
                table.schema,
                compression='snappy',
                flavor='spark',
                append=True  # Важно: append режим
            ) as writer:
                writer.write_table(table)
        
        print(f"Добавлено {len(result_df)} записей в {save_path}")

In [ ]:
def save_csv_results(ids, preds, save_path, chunk_num):
    """Вспомогательная функция для дозаписи в CSV"""
    res_df = pd.DataFrame({
        'event_id': ids,
        'predict': preds
    })
    
    # Если это первый чанк, создаем файл (w), если нет - дописываем (a)
    mode = 'w' if chunk_num == 0 else 'a'
    header = True if chunk_num == 0 else False
    
    res_df.to_csv(save_path, mode=mode, header=header, index=False)

def predict_in_chunks(file: str = None,
                      batchsize: int = 20000,
                      cols_json_file: str = None,
                      return_type: str = 'proba',
                      model = None,
                      save_path: str = None, # Теперь это путь к .csv
                      auto_nan_filler: bool = True
                      ):
    
    if any(v is None for v in [file, cols_json_file, model, save_path]):
        raise AttributeError("Проверь входные аргументы: file, cols_json_file, model, save_path")
    
    with open(cols_json_file, 'r') as f:
        cols = json.load(f)
    
    feature_cols = cols['all']
    id_col = cols['id']
    
    print(f"Обработка файла: {file}")
    
    current_file = pq.ParquetFile(file)
    total_rows = current_file.metadata.num_rows
    total_batches = (total_rows + batchsize - 1) // batchsize
    
    for batch_idx, batch in enumerate(current_file.iter_batches(batch_size=batchsize)):
        print(f'Старт {batch_idx + 1}/{total_batches}')
        
        df_chunk = batch.to_pandas()
        # Выбираем фичи и ID
        chunk = df_chunk[feature_cols].copy() 
        id_events = df_chunk[id_col]
        
        if auto_nan_filler:
            # Заполняем NaN и в инты, чтобы CSV был чище
            chunk = chunk.fillna(-1).astype(int)

        if return_type == 'proba':
            preds = model.predict_proba(chunk)[:, 1]
        else:
            preds = model.predict(chunk)
        
        # Сохраняем текущий чанк
        save_csv_results(id_events, preds, save_path, chunk_num=batch_idx)
        print(f'Батч {batch_idx + 1} готов')

    print(f"Готово! Результаты тут: {save_path}")

In [6]:
valid_predict_data_path = '../datasets/joined/valid_data.parquet'
valid_predict_saving_path = '../datasets/validation/validation_result.parquet'

predict_in_chunks(file=valid_predict_data_path,
                  cols_json_file='../datasets/joined/columns_list.json',
                  model=catboost_model,
                  save_path=valid_predict_saving_path)

Обработка файла: ../datasets/joined/valid_data.parquet
Старт 1/3
Батч 1 готов
Старт 2/3
Батч 2 готов
Старт 3/3
Батч 3 готов
Готово! Результаты тут: ../datasets/validation/validation_result.parquet


## Error Analysis

In [7]:
# Кастомная реализация sklearn.average_precision_score для big data
def average_precision_score(predict_path: str,
                            answers_path: str,
                            predict_column: str = 'target',
                            answers_column: str = 'target',
                            id_column: str = 'event_id'
                            ) -> float:
    """
    Расчет Average Precision (AP) метрики на больших данных с использованием Spark.
    
    AP = Σ (R_n - R_{n-1}) * P_n, где:
    - P_n - precision на n-ом пороге
    - R_n - recall на n-ом пороге
    
    predict_path: путь до parquet файла с предсказаниями (вероятности от 0 до 1)
    answers_path: путь до parquet файла с правильными ответами (бинарные метки 0/1)
    predict_column: название колонки с предсказаниями
    answers_column: название колонки с правильными ответами
    id_column: название колонки с идентификатором события
    
    Возвращает float: значение Average Precision метрики
    """
    
    spark = SparkSession.builder \
        .appName("AveragePrecisionCalculatorOptimized") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()

    try:
        # Загружаем и объединяем данные
        predictions = spark.read.parquet(predict_path)
        answers = spark.read.parquet(answers_path)
        
        predict_cols = predictions.columns
        answers_cols = answers.columns
        if id_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {id_column}")
        if id_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {id_column}")
        if predict_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {predict_column}")
        if answers_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {answers_column}")
        
        
        merged = (predictions
                    .select(id_column, predict_column)
                    .join(answers.select(id_column, answers_column),
                        on=id_column,
                        how="inner")
                    .cache())
        
        total_count = merged.count()
        total_positives = merged.filter(col(answers_column) == 1).count()
        
        if total_positives == 0:
            return 0.0
        
        # Определяем окно для сортировки по убыванию предсказаний
        window_spec = Window.orderBy(col(predict_column).desc())
        
        # Добавляем индикаторы и кумулятивные суммы
        ap_data = merged.withColumn(
            "is_positive", col(answers_column)
        ).withColumn(
            "cumulative_tp", sum("is_positive").over(window_spec)
        ).withColumn(
            "row_num", row_number().over(window_spec)
        )
        
        # Расчет precision и recall для каждой строки
        # Используем lag для получения предыдущих значений
        result = ap_data.withColumn(
            "precision", col("cumulative_tp") / col("row_num")
        ).withColumn(
            "prev_tp", lag("cumulative_tp", 1, 0).over(window_spec)
        ).withColumn(
            "recall_diff", 
            (col("cumulative_tp") - col("prev_tp")) / lit(total_positives)
        ).agg(
            sum(col("recall_diff") * col("precision")).alias("average_precision")
        ).collect()[0][0]
        
        merged.unpersist()
        
        return float(result if result is not None else 0.0)
        
    finally:
        spark.stop()

In [8]:
def target_class_mistakes_finder(predict_path: str,
                                 answers_path: str,
                                 predict_column: str = 'target',
                                 answers_column: str = 'target',
                                 id_column: str = 'event_id', 
                                 target_class: int = 1):
    """
    Функция по поиску расхождений предикта с известными данными
    именно с целевым классом - False Negative результаты
    Принимает:
    predict_path - полный путь к parquet файлу предсказания
    answers_path - полный путь к parquet файлу с ответами
    predcit_column - название колонки с лейблами предсказаний
    answers_column - название колонки с лейблами ответов
    id_column - название колонки для вывода
    target_class - фильтр по классу
    """
    # Создаем Spark сессию
    spark = SparkSession.builder \
        .appName("FindFalseNegatives") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()

    try:
        # Загружаем данные и проверяем колонки
        predictions = spark.read.parquet(predict_path)
        answers = spark.read.parquet(answers_path)
        
        predict_cols = predictions.columns
        answers_cols = answers.columns
        
        if id_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {id_column}")
        if id_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {id_column}")
        if predict_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {predict_column}")
        if answers_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {answers_column}")
        
        # Объединяем данные
        merged = (predictions
                  .select(id_column, predict_column)
                  .join(answers.select(id_column, answers_column),
                        on=id_column,
                        how="inner")
                  )
        
        # Определяем условие для поиска ошибок и находим ошибки
        if target_class in [0, 1]:
            # реальный ответ = target class, предсказание иное
            error_condition = (col(answers_column) == target_class) & (col(predict_column) != target_class)
        else:
            raise ValueError("target_class должен быть 0, 1")
        
        errors = merged.filter(error_condition)
        
        # Считаем статистику
        total_count = merged.count()
        mismatch_count = merged.filter(col(predict_column) != col(answers_column)).count()
        error_count = errors.count()
        
        print(f"Всего записей: {total_count}")
        print(f"Всего несоответствий: {mismatch_count}")
        print(f"Найдено целевых ошибок: {error_count}")
        
        if error_count > 0:
            print(f"\nПервые 10 {id_column} с ошибками:")
            errors.select(id_column).show(10)

        return [row[id_column] for row in errors.select(id_column).collect()]
        
    finally:
        spark.stop()
    

In [9]:
# Поиск лучших параметров и отбор полезных признаков (shap?)

In [10]:
target_class_mistakes_finder(predict_path=valid_predict_saving_path,
                             answers_path=valid_predict_data_path)

Py4JJavaError: An error occurred while calling o30.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (TARDIS executor driver): org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:436)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:506)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1(ParquetFileFormat.scala:557)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1$adapted(ParquetFileFormat.scala:549)
	at org.apache.spark.sql.execution.data

## Test Predict (подготовка решения)

In [11]:
train_files = [valid_train_data_path, valid_predict_data_path]

train_in_chunks(files_list=train_files,
                cols_json_file='../datasets/joined/columns_list.json',
                model=catboost_model)

predict_in_chunks(file='../datasets/joined/test_data.parquet',
                  cols_json_file='../datasets/joined/columns_list.json',
                  model=catboost_model,
                  save_path='../submit/submit_2.csv')

В файле ../datasets/joined/train_data.parquet 91891 строк. Это 5 батчей
Старт обработки 1/5 батча
В target батча содержится:
0 - 19959 штук
1 - 41 штук
Старт обучения 1/5 батча
Обучение 1/5 батча завершено
Старт обработки 2/5 батча
В target батча содержится:
0 - 19937 штук
1 - 63 штук
Старт обучения 2/5 батча
Обучение 2/5 батча завершено
Старт обработки 3/5 батча
В target батча содержится:
0 - 19949 штук
1 - 51 штук
Старт обучения 3/5 батча
Обучение 3/5 батча завершено
Старт обработки 4/5 батча
В target батча содержится:
0 - 19944 штук
1 - 56 штук
Старт обучения 4/5 батча
Обучение 4/5 батча завершено
Старт обработки 5/5 батча
В target батча содержится:
0 - 11870 штук
1 - 21 штук
Старт обучения 5/5 батча
Обучение 5/5 батча завершено
В файле ../datasets/joined/valid_data.parquet 49723 строк. Это 3 батчей
Старт обработки 1/3 батча
В target батча содержится:
0 - 19955 штук
1 - 45 штук
Старт обучения 1/3 батча
Обучение 1/3 батча завершено
Старт обработки 2/3 батча
В target батча содержится:

## Submit Creating (Создание сабмита)

In [ ]:
save_submit_path = '../submits/submit.csv'

def create_submit(save_path: str = save_submit_path,
                  take_path: str = '../submits/submit.parquet'):
    result_df = pd.read_parquet(take_path)
    
    row_count = len(result_df)
    if row_count != 633683:
        raise ValueError(f"Представленный датафрейм имеет иное количество строк: {row_count}. Такой сабмит не пройдет. Необходимо 633683 строк")
    
    if list(result_df.columns) != ['event_id', 'predict']:
        raise ValueError(f'Неверные названия столбиков: {result_df.columns}')
    
    with open(save_path, 'w') as file:
        result_df.to_csv(file, sep=',', index=False)

In [ ]:
create_submit()